In [4]:
import os
import numpy as np
import pandas as pd
from typing import List, Tuple, Optional
from dotenv import load_dotenv
import torch


# Clustering
from sklearn.cluster import MiniBatchKMeans
from umap import UMAP

# BERTopic imports
from bertopic import BERTopic
from bertopic.representation import (
    KeyBERTInspired,
    PartOfSpeech,
    MaximalMarginalRelevance,
    OpenAI
)

# OpenAI
import openai

# Visualization
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots


from bertopic import BERTopic
import tiktoken
from bertopic.representation import (
        KeyBERTInspired,
        PartOfSpeech,
        MaximalMarginalRelevance,
        OpenAI
    )

load_dotenv()
client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [2]:

# Enable CUDA debugging only if needed
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # Use first GPU

# PyTorch GPU optimization
torch.cuda.empty_cache()
#torch.cuda.set_per_process_memory_fraction(0.95)  # Use 80% of 8.4GB = 6.7GB

print(f"\nCUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    total_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Total GPU Memory: {total_memory:.1f} GB")
   


CUDA Available: True
Device: NVIDIA RTX A4000 Laptop GPU
CUDA Version: 12.8
Total GPU Memory: 8.4 GB


In [3]:
#print("Loading LRFAF dataset from Hugging Face...")
#dataset = load_dataset("regicid/LRFAF")

# Convert to pandas DataFrame
df = pd.read_csv("/home/robin/Code_repo/psycholinguistic2125/JADT_rap_fr/data/20251126_cleaned_verses_lrfaf_corpus.csv")
df = df[(df.year > 1991) & (df.year < 2024)]
print(f"Dataset loaded: {len(df)} verses/songs")
print(f"Columns: {df.columns.tolist()}")
print(f"Date range: {df['year'].min()} to {df['year'].max()}")


Dataset loaded: 120662 verses/songs
Columns: ['artist', 'title', 'year', 'lyrics', 'lyrics_cleaned', 'born_in_france', 'url', 'birthdate_artist', 'age_artist']
Date range: 1992.0 to 2023.0


In [16]:
#to train embedings : docs = df['lyrics_cleaned'].astype(str).tolist()
"""embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")

embeddings = embedding_model.encode(
    docs,
    batch_size=128,               # adjust to your GPU/CPU
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True    # cosine similarity-friendly
)

print("Embeddings shape:", embeddings.shape)  # (36000, 768) etc.

# Save to disk
np.save("models/verses_mpnet_base_embeddings.npy", embeddings)"""


'embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")\n\nembeddings = embedding_model.encode(\n    docs,\n    batch_size=128,               # adjust to your GPU/CPU\n    show_progress_bar=True,\n    convert_to_numpy=True,\n    normalize_embeddings=True    # cosine similarity-friendly\n)\n\nprint("Embeddings shape:", embeddings.shape)  # (36000, 768) etc.\n\n# Save to disk\nnp.save("models/verses_mpnet_base_embeddings.npy", embeddings)'

In [5]:
from bertopic.representation._base import BaseRepresentation

class DummyEmbeddings(BaseRepresentation):
    """
    Dummy embedding class that returns None (tells BERTopic to use 
    pre-computed embeddings from fit_transform).
    """
    def extract_topics(self, topic_model, documents, c_tf_idf, topics):
        return topics

    def __call__(self, topic_model, documents, c_tf_idf, topics):
        return topics


def create_topic_model_with_precomputed_embeddings(
    docs,
    embeddings,
    umap_model,
    cluster_model,
    seed_topic_list,
    openai_client,
):
    """
    Create BERTopic with pre-computed embeddings (no embedding_model needed).

    Args:
        docs: List of documents
        embeddings: Pre-computed embeddings (n_docs, embedding_dim)
        umap_model: Fitted UMAP model
        cluster_model: Fitted clustering model
        openai_client: OpenAI client
        n_topics: Number of topics

    Returns:
        topic_model: Fitted BERTopic model
        topics: Topic assignments
        probs: Topic probabilities
    """

    

    # Representation models
    main_representation = None
    aspect_model1 = PartOfSpeech("fr_core_news_sm")
    aspect_model2 = MaximalMarginalRelevance(diversity=0.5)


    openai_representation = OpenAI(
        openai_client,
        model="gpt-4o-mini",
        chat=True,
        nr_docs=10,
        delay_in_seconds=2,
        doc_length=100,
        diversity = 0.3,
        tokenizer= tiktoken.encoding_for_model("gpt-4o-mini"),
        prompt="""J'ai un sujet qui contient les documents suivants :\n[DOCUMENTS]
        Le sujet est décrit par les mots-clés suivants : [KEYWORDS]
        En te basant sur les informations ci-dessus, peux-tu donner un court label du sujet en relation avec la culture hip-hop française 
        Réponds en une seule ligne, avec un titre concis (2-4 mots) en français.
        Exemple : "Amour et Relations", "Rue et Criminalité", "Politique Française"""
    )

    representation_model = {
        "Main": main_representation,
        "spacy_POS": aspect_model1,
        "MMR": aspect_model2,
        "OpenAI": openai_representation,
    }

    # Create and fit BERTopic
    topic_model = BERTopic(
        hdbscan_model=cluster_model,
        umap_model=umap_model,
        embedding_model=None,  # KEY: No embedding model
        language="french",
        low_memory=True,
        representation_model=representation_model,
        seed_topic_list=seed_topic_list,
    )

    topics, probs = topic_model.fit_transform(docs, embeddings=embeddings)

    return topic_model, topics, probs

In [7]:
# 1. Load data
print("Loading data...")
docs = df['lyrics_cleaned'].astype(str).tolist()
timestamps = df['year'].astype(int).astype(str).tolist()
embeddings = np.load("models/verses_mpnet_base_embeddings.npy")

# 2. Setup models
print("Setting up models...")
load_dotenv()
openai_client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

umap_model = UMAP(
    n_neighbors=15, n_components=15, min_dist=0.0,
    metric='cosine', random_state=42
)

cluster_model = MiniBatchKMeans(
    n_clusters=25, random_state=42, batch_size=4096
)

seed_topic_list=  [
    ["rap", "flow", "album", "mc", "prod", "écouter", "street", "couplet", "lourd", "rappe", "beat", "top", "dj", "découper", "disque", "classique", "français", "freestyle", "hit", "boss", "hip_hop", "titre", "instru", "nouveau", "radio", "mixtape", "cd", "rime", "niveau", "kicker"],
    ["mec", "gars", "mic", "style", "équipe", "taper", "xxx", "bouger", "clique", "claque", "braquer", "pro", "débarquer", "foutre", "check", "mac", "kick", "type", "place", "micro", "attaque", "flingue", "phase", "dingue", "jouer", "frapper", "péter", "bac", "chaud", "mettre"],

    ["pute", "sucer", "bite", "baiser", "nique", "fils", "cul", "chat", "salope", "bande", "tasse", "mère", "niquer", "meuf", "pé", "lécher", "enculer", "chien", "putain", "gros", "pédé", "puer", "buzz", "bise", "queue", "bâtard", "suceur", "tass", "enculé", "avaler"],

    ["fumer", "beuh", "rouler", "joint", "vert", "billet", "violett", "jaune", "dé", "weed", "pilon", "teh", "kush", "foncer", "jack", "mula", "bleu", "vodka", "mauve", "allumer", "rouge", "verre", "clope", "marron", "bois", "og", "blunt", "mélange", "fumée", "sirop"],

    ["euro", "zone", "payer", "monnaie", "minuit", "bendo", "midi", "terrain", "igo", "vendre", "bibi", "condé", "charbonner", "biff", "bolide", "aller", "mouler", "bicrave", "four", "détailler", "hess", "rentrer", "tess", "binks", "sacoche", "sortir", "poto", "rrain", "gros", "kilo"], # Classe 12

    ["han", "hey", "yah", "mmh", "woh", "gang", "louis", "brr", "wow", "pah", "ekip", "grr", "sku", "wouh", "okay", "uh", "wah", "woah", "drip", "babe", "gucci", "vuitton", "jojo", "nan", "hun", "hmm", "lean", "nah", "pussy", "paw"], # Classe 3

    ["homme", "guerre", "peuple", "pays", "loi", "paix", "système", "politique", "droit", "liberté", "état", "violence", "combat", "justice", "monde", "pauvre", "subir", "humain", "libre", "humanité", "victime", "religion", "culture", "diviser", "afrique", "enfant", "coupable", "médias", "société", "riche"], # Classe 1

    ["vie", "temps", "ami", "perdre", "chose", "changer", "chance", "penser", "vivre", "choix", "oublier", "famille", "donner", "dur", "meilleur", "confiance", "moment", "regretter", "fois", "passer", "compte", "avis", "rendre", "réussir", "maman", "avancer", "raconter", "essayer", "gagner", "grandir"], # Classe 5

    ["aimer", "gens", "mentir", "vraiment", "amour", "comprendre", "semblant", "plaire", "détester", "sentiment", "autrement", "écouter", "femme", "compliqué", "besoin", "franchement", "heureux", "vrai", "accord", "chose", "fille", "aide", "différent", "chanson", "décevoir", "sincère", "tromper", "avouer", "gentil", "préférer"], # Classe 6

    ["lumière", "lune", "étoile", "noir", "ombre", "plume", "briller", "bout", "éteindre", "sombre", "gris", "éclairer", "clair", "tunnel", "route", "blanc", "long", "écrire", "brume", "univers", "souffle", "ciel", "brouillard", "bougie", "illuminer", "toile", "décrocher", "vent", "espoir", "doute"], # Classe 10

    ["oeil", "ciel", "coeur", "regarder", "tomber", "terre", "ouvrir", "ange", "corps", "enfer", "voir", "regard", "cieux", "fleur", "aide", "vide", "mer", "porte", "fermer", "beau", "sourire", "paradis", "visage", "pleurer", "froid", "fenêtre", "emmener", "sol", "pluie", "prier"], # Classe 11

    ["nuit", "dormir", "sommeil", "réveiller", "lever", "soleil", "soir", "matin", "cauchemard", "endormir", "coucher", "rêve", "lit", "réveil", "insomnie", "jour", "heure", "drap", "journée", "ennui", "tôt", "veille", "sonner", "ennuyer", "revoir", "rêver", "bras", "sommet", "lèvre", "oreille"]
    ]

# 3. Create and fit model
print("Fitting BERTopic...")
topic_model, topics, probs = create_topic_model_with_precomputed_embeddings(docs = docs, 
                                                                            embeddings=embeddings,
                                                                              umap_model  = umap_model, 
                                                                              cluster_model = cluster_model,
                                                                              seed_topic_list = seed_topic_list,
                                                                              openai_client=openai_client)


Loading data...
Setting up models...
Fitting BERTopic...


In [8]:
def get_topic_label(topic_model, topic_id, use_custom_labels=True):

    """Get label for a topic."""
    
    if use_custom_labels and hasattr(topic_model, 'custom_labels_') and topic_model.custom_labels_:
        return topic_model.custom_labels_[topic_id]
    
    try:
        topic_info = topic_model.get_topic_info()
        name = topic_info[topic_info['Topic'] == topic_id]['Name'].values
        if len(name) > 0:
            return name[0]
    except:
        pass
    
    try:
        keywords = topic_model.get_topic(topic_id)
        if keywords:
            return ', '.join([w[0] for w in keywords[:3]])
    except:
        pass
    
    return f"Topic {topic_id}"

def clean_label(x):
    """Convert list to clean string"""
    if isinstance(x, list):
        x = x[0] if len(x) > 0 else str(x)
    x = str(x).strip('[]"\'')
    return x

In [9]:
print("=" * 80)
print("STEP 1: PREPARE DATA WITH TOPICS AND METADATA")
print("=" * 80)

# Add topic assignments and metadata to dataframe
df['topic'] = topics
df['year'] = df['year'].astype(int)
df['artist'] = df['artist'].astype(str)

print(f"✓ Loaded {len(df)} documents")
print(f"✓ Topics: {sorted(df['topic'].unique())}")
print(f"✓ Years: {sorted(df['year'].unique())}")
print(f"✓ Artists: {df['artist'].nunique()}")

# Verify no -1 topics (noise cluster)
print(f"✓ Noise documents (-1): {(df['topic'] == -1).sum()}")

STEP 1: PREPARE DATA WITH TOPICS AND METADATA
✓ Loaded 120662 documents
✓ Topics: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24)]
✓ Years: [np.int64(1992), np.int64(1993), np.int64(1994), np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
✓ Artists: 606
✓ Noise documents (-1): 0


In [10]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,spacy_POS,MMR,OpenAI,Representative_Docs
0,0,8126,0_la_les_le_pas,"[la, les, le, pas, est, de, dans, des, on, un]","[fais, tout, fait, veux, gros, ton, vie, toi, ...","[la, les, le, pas, est, de, dans, des, on, un,...","[""Violence et Résilience""]","[Ok Booska-p, Ripro, 1er juin dans les bacs, h..."
1,1,7120,1_les_la_on_de,"[les, la, on, de, pas, est, le, des, qu, dans]","[tu, mon, fait, tous, ton, es, gros, toi, sais...","[les, la, on, de, pas, est, le, des, qu, dans,...","[""Vie Urbaine et Résilience""]","[C'est le plus grand hagar qui gagne, on jalou..."
2,2,7034,2_rap_le_de_est,"[rap, le, de, est, les, la, pas, et, un, on]","[rap, tout, même, tous, ton, veux, monde, musi...","[rap, le, de, est, les, la, pas, et, un, on, j...",[Rap et Identité Sociale],"[Il m'faut des filons et un contrat, qui a mér..."
3,3,6667,3_la_est_le_les,"[la, est, le, les, pas, de, on, dans, des, en]","[fait, ta, ton, toi, même, temps, rap, oui, vo...","[la, est, le, les, pas, de, on, dans, des, en,...","[""Identité et Résilience""]",[KIKESA - SOLO 1 (Lyrics)\n\nChaque semaine un...
4,4,5822,4_je_ai_de_que,"[je, ai, de, que, la, le, pas, est, et, me]","[vie, temps, fait, monde, tous, seul, jour, tê...","[je, ai, de, que, la, le, pas, est, et, me, qu...","[""Souffrance et Créativité""]","[J'rappe avec le coeur et les tripes, j'rappe ..."
5,5,5623,5_la_les_est_le,"[la, les, est, le, de, pas, dans, tu, un, des]","[suis, fais, ouais, veux, ton, as, faut, tous,...","[la, les, est, le, de, pas, dans, tu, un, des,...","[""Vie de Rue et Résilience""]",[L'instru elle est douce et mélancolique as-fu...
6,6,5513,6_de_les_la_on,"[de, les, la, on, est, le, et, des, qu, pas]","[guerre, tous, tout, monde, même, fait, bien, ...","[de, les, la, on, est, le, et, des, qu, pas, q...",[Violence et Résistance],[C’est ici qu’ça s’passe pour tous ceux qui re...
7,7,5462,7_pas_de_la_il,"[pas, de, la, il, les, est, le, qu, ai, un]","[vie, tous, temps, gens, mal, mère, jour, vois...","[pas, de, la, il, les, est, le, qu, ai, un, et...","[""Vulgarité et Résilience""]","[Qui a dit qu’c'est mort dans le 13, c’est un ..."
8,8,5179,8_je_tu_moi_que,"[je, tu, moi, que, toi, pas, ai, amour, de, est]","[pas, amour, tout, cœur, vie, ton, as, tes, fa...","[je, tu, moi, que, toi, pas, ai, amour, de, es...","[""Amour et Désillusion""]",[Ok. Je ne t'ai jamais aimé. Est-ce de ta faut...
9,9,5177,9_la_les_le_est,"[la, les, le, est, de, dans, on, pas, des, un]","[tu, gang, ça, moi, tout, tous, gros, rap, mêm...","[la, les, le, est, de, dans, on, pas, des, un,...","[""Expression Urbaine""]",[Kanoé\nIntro\nEh\nEh\nMesah à la prod\nFreest...


In [11]:
print("\n" + "=" * 80)
print("STEP 2: GET TOPIC LABELS (OpenAI)")
print("=" * 80)

# Get topic info and extract labels
topic_info = topic_model.get_topic_info()
print("Topic labels from model:")
print(topic_info[['Topic', 'OpenAI']].head(25))

# Set custom labels
openai_labels = dict(zip(topic_info['Topic'], topic_info['OpenAI']))
topic_model.set_topic_labels(openai_labels)

print(f"\n✓ Set {len(openai_labels)} OpenAI labels")

# Create mapping for display
topic_name_map = {row['Topic']:  clean_label(row['OpenAI']) for _, row in topic_info.iterrows()}
print(f"\nTopic name mapping created")




STEP 2: GET TOPIC LABELS (OpenAI)
Topic labels from model:
    Topic                                OpenAI
0       0            ["Violence et Résilience"]
1       1         ["Vie Urbaine et Résilience"]
2       2             [Rap et Identité Sociale]
3       3            ["Identité et Résilience"]
4       4          ["Souffrance et Créativité"]
5       5          ["Vie de Rue et Résilience"]
6       6              [Violence et Résistance]
7       7           ["Vulgarité et Résilience"]
8       8              ["Amour et Désillusion"]
9       9                ["Expression Urbaine"]
10     10              [Solitude et Résilience]
11     11          ["Vie Urbaine et Rébellion"]
12     12            ["Expression et Identité"]
13     13           [Mort et Quête Identitaire]
14     14          ["Vie Urbaine et Nostalgie"]
15     15                ["Amour et Déception"]
16     16            ["Résilience et Identité"]
17     17              [Identité et Résistance]
18     18  ["Identité et Rel

In [12]:
print("\n" + "=" * 80)
print("STEP 3: ANALYZE TOP ARTIST PER TOPIC")
print("=" * 80)

topic_artist_analysis = []

for topic_id in sorted(df['topic'].unique()):
    if topic_id == -1:
        continue  # Skip noise

    topic_df = df[df['topic'] == topic_id]

    # Get top artist
    artist_counts = topic_df['artist'].value_counts()
    top_artist = artist_counts.index[0]
    top_artist_count = artist_counts.iloc[0]
    top_artist_pct = (top_artist_count / len(topic_df)) * 100

    # Get topic label
    topic_label = topic_name_map.get(topic_id, f"Topic {topic_id}")

    # Get diversity
    unique_artists = topic_df['artist'].nunique()
    doc_count = len(topic_df)

    analysis = {
        'Topic': topic_id,
        'Label': topic_label,
        'Total_Docs': doc_count,
        'Unique_Artists': unique_artists,
        'Top_Artist': top_artist,
        'Top_Artist_Count': top_artist_count,
        'Top_Artist_Pct': top_artist_pct,
        'Top_3_Artists': ', '.join(artist_counts.head(3).index.tolist()),
    }

    topic_artist_analysis.append(analysis)

artist_df = pd.DataFrame(topic_artist_analysis)
artist_df = artist_df.sort_values('Total_Docs', ascending=False)

print("\nTop Artist per Topic:")
print(artist_df.to_string())
artist_df.to_csv('topic_top_artists.csv', index=False)
print("\n✓ Saved: topic_top_artists.csv")

print(f"Label type: {artist_df['Label'].dtype}")
print(f"Sample: '{artist_df['Label'].iloc[0]}'")


STEP 3: ANALYZE TOP ARTIST PER TOPIC

Top Artist per Topic:
    Topic                             Label  Total_Docs  Unique_Artists       Top_Artist  Top_Artist_Count  Top_Artist_Pct                       Top_3_Artists
0       0            Violence et Résilience        8126             500              JuL               259        3.187300                     JuL, Kekra, PLK
1       1         Vie Urbaine et Résilience        7120             500              JuL               195        2.738764               JuL, La Fouine, Booba
2       2           Rap et Identité Sociale        7034             496            Rohff               139        1.976116            Rohff, Disiz, Busta Flex
3       3            Identité et Résilience        6667             492              JuL               231        3.464827                   JuL, Naps, Alonzo
4       4          Souffrance et Créativité        5822             489          Georgio               115        1.975266                 Georg

In [13]:
print("\n" + "=" * 80)
print("STEP 4: TEMPORAL EVOLUTION ANALYSIS")
print("=" * 80)

# Create temporal matrix: Topics x Years
years = sorted(df['year'].unique())
topics_list = sorted([t for t in df['topic'].unique() if t != -1])

temporal_data = []

for topic_id in topics_list:
    for year in years:
        count = len(df[(df['topic'] == topic_id) & (df['year'] == year)])
        temporal_data.append({
            'Topic': topic_id,
            'Label': topic_name_map.get(topic_id, f"Topic {topic_id}"),
            'Year': year,
            'Count': count
        })

temporal_df = pd.DataFrame(temporal_data)
temporal_df['Label'] = temporal_df['Label'].apply(
    lambda x: ' '.join(map(str, x)) if isinstance(x, list) else str(x)
)

print(f"\n✓ Created temporal matrix: {len(topics_list)} topics × {len(years)} years")
print(f"\nSample temporal data:")
print(temporal_df[temporal_df['Topic'] == 0].head(10))

print("\n" + "=" * 80)
print("STEP 5: APPEARANCE/DISAPPEARANCE ANALYSIS")
print("=" * 80)

appearance_analysis = []

for topic_id in topics_list:
    topic_temporal = temporal_df[temporal_df['Topic'] == topic_id]

    # Find first appearance
    first_year = topic_temporal[topic_temporal['Count'] > 0]['Year'].min()

    # Find last appearance
    last_year = topic_temporal[topic_temporal['Count'] > 0]['Year'].max()

    # Peak year
    peak_row = topic_temporal[topic_temporal['Count'] == topic_temporal['Count'].max()]
    peak_year = peak_row['Year'].iloc[0] if len(peak_row) > 0 else None
    peak_count = peak_row['Count'].iloc[0] if len(peak_row) > 0 else 0

    # Total documents
    total_docs = topic_temporal['Count'].sum()

    # Trend (comparing first half vs second half)
    mid_year = (first_year + last_year) / 2 if first_year and last_year else None

    if mid_year:
        first_half = topic_temporal[topic_temporal['Year'] < mid_year]['Count'].sum()
        second_half = topic_temporal[topic_temporal['Year'] >= mid_year]['Count'].sum()
        trend = "↑ Rising" if second_half > first_half else ("↓ Declining" if second_half < first_half else "→ Stable")
    else:
        trend = "Unknown"

    appearance_analysis.append({
        'Topic': topic_id,
        'Label': topic_name_map.get(topic_id, f"Topic {topic_id}"),
        'First_Appearance': first_year,
        'Last_Appearance': last_year,
        'Lifespan_Years': (last_year - first_year + 1) if first_year and last_year else 0,
        'Peak_Year': peak_year,
        'Peak_Count': peak_count,
        'Total_Documents': total_docs,
        'Trend': trend,
    })

appearance_df = pd.DataFrame(appearance_analysis)
appearance_df = appearance_df.sort_values('Total_Documents', ascending=False)

print("\nAppearance/Disappearance Analysis:")
print(appearance_df.to_string())

# Save for reference
appearance_df.to_csv('topic_temporal_evolution.csv', index=False)
print("\n✓ Saved: topic_temporal_evolution.csv")


STEP 4: TEMPORAL EVOLUTION ANALYSIS

✓ Created temporal matrix: 25 topics × 32 years

Sample temporal data:
   Topic                   Label  Year  Count
0      0  Violence et Résilience  1992      3
1      0  Violence et Résilience  1993     10
2      0  Violence et Résilience  1994      6
3      0  Violence et Résilience  1995     33
4      0  Violence et Résilience  1996     23
5      0  Violence et Résilience  1997     33
6      0  Violence et Résilience  1998     68
7      0  Violence et Résilience  1999     54
8      0  Violence et Résilience  2000     60
9      0  Violence et Résilience  2001     58

STEP 5: APPEARANCE/DISAPPEARANCE ANALYSIS

Appearance/Disappearance Analysis:
    Topic                             Label  First_Appearance  Last_Appearance  Lifespan_Years  Peak_Year  Peak_Count  Total_Documents     Trend
0       0            Violence et Résilience              1992             2023              32       2020         777             8126  ↑ Rising
1       1       

In [39]:
print("\n" + "=" * 80)
print("STEP 6: THEME REPLACEMENT ANALYSIS")
print("=" * 80)

# Analyze which themes are rising vs declining
rising_themes = appearance_df[appearance_df['Trend'] == '↑ Rising'].sort_values('Total_Documents', ascending=False)
declining_themes = appearance_df[appearance_df['Trend'] == '↓ Declining'].sort_values('Total_Documents', ascending=False)
stable_themes = appearance_df[appearance_df['Trend'] == '→ Stable'].sort_values('Total_Documents', ascending=False)

print("\n🔴 DECLINING THEMES (Being replaced):")
if len(declining_themes) > 0:
    for _, row in declining_themes.head(5).iterrows():
        print(f"  - {row['Label'][:50]} ({row['First_Appearance']}-{row['Last_Appearance']}) - {row['Total_Documents']} docs")
else:
    print("  None")

print("\n🟢 RISING THEMES (New/Growing):")
if len(rising_themes) > 0:
    for _, row in rising_themes.head(5).iterrows():
        print(f"  - {row['Label'][:50]} ({row['First_Appearance']}-{row['Last_Appearance']}) - {row['Total_Documents']} docs")
else:
    print("  None")

print("\n🟡 STABLE THEMES (Consistent):")
if len(stable_themes) > 0:
    for _, row in stable_themes.head(5).iterrows():
        print(f"  - {row['Label'][:50]} ({row['First_Appearance']}-{row['Last_Appearance']}) - {row['Total_Documents']} docs")
else:
    print("  None")


STEP 6: THEME REPLACEMENT ANALYSIS

🔴 DECLINING THEMES (Being replaced):
  None

🟢 RISING THEMES (New/Growing):
  - Vie Urbaine et Luttes (1992-2023) - 8126 docs
  - Vie de Quartier (1992-2023) - 7120 docs
  - Identité et Authenticité (1992-2023) - 7034 docs
  - Identité et Résilience (1992-2023) - 6667 docs
  - Souffrance et Résilience (1992-2023) - 5822 docs

🟡 STABLE THEMES (Consistent):
  None


## Visulisation

In [14]:
topic_model.visualize_topics()

In [48]:
topic_model.visualize_heatmap()

In [15]:
print("=" * 80)
print("VISUALIZATION 1: TIMELINE OF ALL TOPICS")
print("=" * 80)

# 1. Timeline - All topics over time
fig1 = go.Figure()

# Sort by total documents for better legend ordering
topic_order = temporal_df.groupby('Label')['Count'].sum().sort_values(ascending=False).index.tolist()

for topic_label in topic_order:
    topic_data = temporal_df[temporal_df['Label'] == topic_label]

    fig1.add_trace(go.Scatter(
        x=topic_data['Year'],
        y=topic_data['Count'],
        mode='lines+markers',
        name=topic_label[:40],  # Truncate long labels
        line=dict(width=2),
        marker=dict(size=6)
    ))

fig1.update_layout(
    title="<b>Topic Evolution Over Time</b><br><sub>Frequency of documents per topic per year</sub>",
    xaxis_title="Year",
    yaxis_title="Document Count",
    hovermode='x unified',
    template="plotly_white",
    height=700,
    width=1400,
    showlegend=True,
    legend=dict(x=1.02, y=1, xanchor='left', yanchor='top')
)

fig1.write_html("viz_01_timeline_all_topics.html")
print("✓ Saved: viz_01_timeline_all_topics.html")

fig1

VISUALIZATION 1: TIMELINE OF ALL TOPICS
✓ Saved: viz_01_timeline_all_topics.html


In [49]:
# ============ VIZ 9: STACKED AREA - TOPIC SHARE % ============
# Get raw counts per (year, topic)
topic_year = temporal_df.groupby(['Year', 'Label'])['Count'].sum().reset_index()

# Pivot: rows = years, columns = topics
pivot = topic_year.pivot_table(
    index='Year',
    columns='Label',
    values='Count',
    fill_value=0
)

# Normalize: each row = 100%
share = pivot.div(pivot.sum(axis=1), axis=0) * 100

# Plot
fig9 = px.area(
    share.reset_index(),
    x='Year',
    y=share.columns[1:],  # All topics (skip Year)
    title="<b>Topic Share Evolution</b><br><sub>What % of corpus is each topic? (sum = 100% per year)</sub>",
    labels={'Year': 'Year', 'value': 'Share (%)', 'variable': 'Topic'},
)

fig9.update_layout(
    yaxis=dict(
        title="Topic Share (%)",
        tickformat=".0f",
        range=[0, 100]
    ),
    xaxis_title="Year",
    hovermode='x unified',
    height=700,
    width=1400,
    template="plotly_white",
    legend=dict(
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.02
    )
)

fig9.write_html("viz_09_stacked_area_share.html")
print("✓ Saved: viz_09_stacked_area_share.html")
fig9

✓ Saved: viz_09_stacked_area_share.html


In [50]:
# ============ VIZ 10: CUMULATIVE DOCUMENTS ============
# Sort by year
topic_year_sorted = temporal_df.sort_values('Year')

# Pivot
pivot_cumul = topic_year_sorted.pivot_table(
    index='Year',
    columns='Label',
    values='Count',
    fill_value=0
)

# Cumulative sum
cumulative = pivot_cumul.cumsum()

# Plot
fig10 = go.Figure()

for topic in cumulative.columns:
    fig10.add_trace(go.Scatter(
        x=cumulative.index,
        y=cumulative[topic],
        mode='lines',
        name=topic[:40],
        stackgroup='one',  # Stacked area
        fillcolor=None,
        line=dict(width=1)
    ))

fig10.update_layout(
    title="<b>Cumulative Document Count</b><br><sub>Total documents per topic over time (stacked)</sub>",
    xaxis_title="Year",
    yaxis_title="Cumulative Documents",
    hovermode='x unified',
    height=700,
    width=1400,
    template="plotly_white",
    legend=dict(
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.02
    )
)

fig10.write_html("viz_10_cumulative_documents.html")
print("✓ Saved: viz_10_cumulative_documents.html")
fig10

✓ Saved: viz_10_cumulative_documents.html


In [41]:
print("\n" + "=" * 80)
print("VISUALIZATION 2: HEATMAP OF TOPIC INTENSITY OVER TIME")
print("=" * 80)

# 2. Heatmap - Topics × Years with intensity
pivot_data = temporal_df.pivot_table(
    values='Count', 
    index='Label', 
    columns='Year', 
    aggfunc='sum',
    fill_value=0
)

# Normalize by year to see relative importance
pivot_normalized = pivot_data.div(pivot_data.sum(axis=0), axis=1) * 100

fig2 = go.Figure(data=go.Heatmap(
    z=pivot_normalized.values,
    x=pivot_normalized.columns,
    y=pivot_normalized.index,
    colorscale='YlOrRd',
    colorbar=dict(title='% of Year')
))

fig2.update_layout(
    title="<b>Topic Distribution Heatmap Over Time</b><br><sub>Normalized by year (% of documents)</sub>",
    xaxis_title="Year",
    yaxis_title="Topic",
    height=600,
    width=1200,
    template="plotly_white"
)

fig2.write_html("viz_02_heatmap_topics_years.html")
print("✓ Saved: viz_02_heatmap_topics_years.html")

fig2


VISUALIZATION 2: HEATMAP OF TOPIC INTENSITY OVER TIME
✓ Saved: viz_02_heatmap_topics_years.html


In [51]:
# ============ VIZ 11: TOPIC DOMINANCE HEATMAP (RANKED) ============
# Get top N topics per year
topic_year = temporal_df.groupby(['Year', 'Label'])['Count'].sum().reset_index()
pivot_rank = topic_year.pivot_table(
    index='Year',
    columns='Label',
    values='Count',
    fill_value=0
)

# Rank topics by total volume (sort columns)
topic_totals = pivot_rank.sum().sort_values(ascending=False)
pivot_rank = pivot_rank[topic_totals.index[:15]]  # Top 15 topics

# Heatmap
fig11 = go.Figure(data=go.Heatmap(
    z=pivot_rank.values,
    x=pivot_rank.columns,
    y=pivot_rank.index,
    colorscale='Blues',
    colorbar=dict(title='Documents')
))

fig11.update_layout(
    title="<b>Topic Dominance Over Time</b><br><sub>Document count heatmap - darker = more documents</sub>",
    xaxis_title="Topic (Top 15 by volume)",
    yaxis_title="Year",
    height=600,
    width=1200,
    template="plotly_white",
    xaxis=dict(tickangle=-45)
)

fig11.write_html("viz_11_dominance_heatmap.html")
print("✓ Saved: viz_11_dominance_heatmap.html")
fig11

✓ Saved: viz_11_dominance_heatmap.html


In [54]:
 #============ VIZ 14: TOPIC LIFECYCLE BUBBLE ============
fig14 = go.Figure()

for _, row in appearance_df.iterrows():
    fig14.add_trace(go.Scatter(
        x=[row['Peak_Year']],
        y=[row['Label']],
        mode='markers',
        marker=dict(
            size=row['Total_Documents'] / 20,  # Bubble size = total docs
            color=row['Lifespan_Years'],  # Color = lifespan
            colorscale='Viridis',
            showscale=False,
            line=dict(width=2, color='white')
        ),
        text=f"<b>{row['Label']}</b><br>" +
             f"Peak Year: {row['Peak_Year']}<br>" +
             f"Lifespan: {row['Lifespan_Years']} years<br>" +
             f"Total Docs: {row['Total_Documents']}<br>" +
             f"Trend: {row['Trend']}",
        hovertemplate='%{text}<extra></extra>',
        name=''
    ))

fig14.update_layout(
    title="<b>Topic Lifecycle Overview</b><br><sub>Bubble size = total documents, position = peak year, color = lifespan</sub>",
    xaxis_title="Year of Peak Popularity",
    yaxis_title="Topic",
    height=700,
    width=1200,
    template="plotly_white",
    showlegend=False,
    hovermode='closest'
)

fig14.write_html("viz_14_lifecycle_bubble.html")
print("✓ Saved: viz_14_lifecycle_bubble.html")
fig14

✓ Saved: viz_14_lifecycle_bubble.html


In [42]:
# 3. Stacked area chart
temporal_sorted = temporal_df.sort_values('Count', ascending=False)
topic_labels_sorted = temporal_df.groupby('Label')['Count'].sum().sort_values(ascending=False).index.tolist()

fig3 = go.Figure()

for topic_label in topic_labels_sorted:
    topic_data = temporal_df[temporal_df['Label'] == topic_label].sort_values('Year')

    fig3.add_trace(go.Scatter(
        x=topic_data['Year'],
        y=topic_data['Count'],
        mode='lines',
        name=topic_label[:35],
        stackgroup='one',
        fillcolor=None
    ))

fig3.update_layout(
    title="<b>Stacked Topic Composition Over Time</b><br><sub>Absolute volume by topic</sub>",
    xaxis_title="Year",
    yaxis_title="Total Documents",
    hovermode='x unified',
    template="plotly_white",
    height=600,
    width=1200,
    showlegend=True,
    legend=dict(x=1.02, y=1, xanchor='left', yanchor='top')
)

fig3.write_html("viz_03_stacked_area_topics.html")
print("✓ Saved: viz_03_stacked_area_topics.html")
fig3

✓ Saved: viz_03_stacked_area_topics.html


In [43]:
print("\n" + "=" * 80)
print("VISUALIZATION 7: TOP ARTIST DOMINANCE")
print("=" * 80)

# 7. Top artist dominance
artist_dominance = artist_df.sort_values('Top_Artist_Pct', ascending=True).tail(50)

fig7 = go.Figure()

fig7.add_trace(go.Bar(
    y=artist_dominance['Label'].str[:40],
    x=artist_dominance['Top_Artist_Pct'],
    orientation='h',
    name='% of Topic',
    marker=dict(color='#ff7f0e'),
    text=artist_dominance['Top_Artist'].str[:20],
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>' +
                 'Top Artist: %{text}<br>' +
                 'Dominance: %{x:.1f}%<br>' +
                 '<extra></extra>'
))

fig7.update_layout(
    title="<b>Top Artist Dominance per Topic (Top 15)</b><br><sub>What % of topic is represented by #1 artist?</sub>",
    xaxis_title="Top Artist Percentage %",
    yaxis_title="Topic",
    height=600,
    width=1000,
    template="plotly_white",
    showlegend=False
)

fig7.write_html("viz_07_top_artist_dominance.html")
print("✓ Saved: viz_07_top_artist_dominance.html")
fig7


VISUALIZATION 7: TOP ARTIST DOMINANCE
✓ Saved: viz_07_top_artist_dominance.html


In [44]:
print("\n" + "=" * 80)
print("VISUALIZATION 8: LIFESPAN ANALYSIS")
print("=" * 80)

# 8. Lifespan
lifespan_sorted = appearance_df.sort_values('Lifespan_Years', ascending=True).tail(min(15, len(df)))

fig8 = go.Figure()

fig8.add_trace(go.Bar(
    y=lifespan_sorted['Label'].str[:40],
    x=lifespan_sorted['Lifespan_Years'],
    orientation='h',
    marker=dict(color='#2ca02c'),
    text=lifespan_sorted['First_Appearance'].astype(str) + '-' + lifespan_sorted['Last_Appearance'].astype(str),
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>' +
                 'Years Active: %{x}<br>' +
                 'Period: %{text}<br>' +
                 '<extra></extra>'
))

fig8.update_layout(
    title="<b>Topic Lifespan (Top 15)</b><br><sub>How many years was each topic actively discussed?</sub>",
    xaxis_title="Years Active",
    yaxis_title="Topic",
    height=600,
    width=1000,
    template="plotly_white",
    showlegend=False
)

fig8.write_html("viz_08_lifespan.html")
print("✓ Saved: viz_08_lifespan.html")
fig8


VISUALIZATION 8: LIFESPAN ANALYSIS
✓ Saved: viz_08_lifespan.html


In [ ]:
print("\n" + "=" * 80)
print("STEP 3: ANALYZE TOP ARTIST PER TOPIC")
print("=" * 80)

topic_artist_analysis = []

for topic_id in sorted(df['topic'].unique()):
    if topic_id == -1:
        continue  # Skip noise

    topic_df = df[df['topic'] == topic_id]

    # Get top artist
    artist_counts = topic_df['artist'].value_counts()
    top_artist = artist_counts.index[0]
    top_artist_count = artist_counts.iloc[0]
    top_artist_pct = (top_artist_count / len(topic_df)) * 100

    # Get topic label
    topic_label = topic_name_map.get(topic_id, f"Topic {topic_id}")

    # Get diversity
    unique_artists = topic_df['artist'].nunique()
    doc_count = len(topic_df)

    analysis = {
        'Topic': topic_id,
        'Label': topic_label,
        'Total_Docs': doc_count,
        'Unique_Artists': unique_artists,
        'Top_Artist': top_artist,
        'Top_Artist_Count': top_artist_count,
        'Top_Artist_Pct': top_artist_pct,
        'Top_3_Artists': ', '.join(artist_counts.head(3).index.tolist()),
    }

    topic_artist_analysis.append(analysis)

artist_df = pd.DataFrame(topic_artist_analysis)
artist_df = artist_df.sort_values('Total_Docs', ascending=False)

print("\nTop Artist per Topic:")
print(artist_df.to_string())

In [70]:
hierarchical_topics = topic_model.hierarchical_topics(docs)
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics, use_ctfidf = False)

100%|██████████| 99/99 [00:00<00:00, 112.07it/s]


In [71]:
topic_model.get_topic_info()

,Topic,Count,Name,CustomName,Representation,spacy_POS,MMR,OpenAI,Representative_Docs
0,0,2523,0_de_les_ils_on,[Démocratie et Résistance],"[de, les, ils, on, nous, la, et, des, qui, est]","[guerre, leur, tous, monde, même, tout, fait, ...","[de, les, ils, on, nous, la, et, des, qui, est...",[Démocratie et Résistance],[L'époque est telle qu'au moindre blème les go...
1,1,2246,1_nuit_lune_je_la,"[""Lutte Intérieure et Identité""]","[nuit, lune, je, la, dans, le, de, ma, me, les]","[nuit, lune, vie, soleil, ciel, oh, yeux, fait...","[nuit, lune, je, la, dans, le, de, ma, me, les...","[""Lutte Intérieure et Identité""]","[Les yeux fermés, les poings serrés, j'essaie ..."
2,2,2059,2_rap_le_est_de,"[""Authenticité et Résilience""]","[rap, le, est, de, les, la, du, et, un, pas]","[rap, album, tout, ton, veux, tous, même, flow...","[rap, le, est, de, les, la, du, et, un, pas, c...","[""Authenticité et Résilience""]","[Non mais, t'sais, j'ai essayé, moi, de... d'é..."
3,3,1966,3_paris_de_la_les,"[""Vies de Quartier""]","[paris, de, la, les, est, un, le, en, des, et]","[euros, tu, vie, veux, ville, tous, fais, ton,...","[paris, de, la, les, est, un, le, en, des, et,...","[""Vies de Quartier""]","[C'est l'histoire de ma vie, so\nOuais, ouais,..."
4,4,1862,4_ka_mwen_an_la,"[""Vie de Quartier""]","[ka, mwen, an, la, pa, ou, nou, on, le, les]","[ka, mwen, an, pa, money, you, tu, fais, fè, y...","[ka, mwen, an, la, pa, ou, nou, on, le, les, e...","[""Vie de Quartier""]",[An ka tiré dan l'ta mè an pa ka réfléchi\nSi ...
...,...,...,...,...,...,...,...,...,...
95,95,436,95_étais_ai_école_des,"[""École et Adolescence""]","[étais, ai, école, des, de, était, les, pas, u...","[école, ans, fait, vie, époque, mère, tous, co...","[étais, ai, école, des, de, était, les, pas, u...","[""École et Adolescence""]",[Haha! Davodka! La confession! Les maux de la ...
96,96,433,96_où_non_pourquoi_bla,"[""Questions et Indécisions""]","[où, non, pourquoi, bla, allô, combien, quoi, ...","[tu, oh, sais, veux, nia, ouais, woh, fais, to...","[où, non, pourquoi, bla, allô, combien, quoi, ...","[""Questions et Indécisions""]","[Uh, check, uh\nNon, non, non, pas ça\nNon, no..."
97,97,414,97_ekip_sku_chen_ze,"[""Fierté et Identité Urbaine""]","[ekip, sku, chen, ze, zga, comme, endurci, nég...","[ekip, sku, ze, zga, endurci, négro, goddamn, ...","[ekip, sku, chen, ze, zga, comme, endurci, nég...","[""Fierté et Identité Urbaine""]","[Ahah, ekip ekip, MMS, LDO\nNRM, 667, pétasse,..."
98,98,166,98_lalalala_laï_lala_lalala,[Musique et Émotions],"[lalalala, laï, lala, lalala, la, lalalalala, ...","[lalalala, laï, lala, lalalalala, génie, oh, m...","[lalalala, laï, lala, lalala, la, lalalalala, ...",[Musique et Émotions],[J'fais comme si toi et moi c'était pas comme ...


In [ ]:
def apply_topics_over_time_to_hierarchical(
    topic_model, docs, timestamps, hierarchical_topics=None,
    global_tuning=True, evolution_tuning=True
):
    """Fixed for YOUR binary hierarchy structure"""
    
    if hierarchical_topics is None:
        hierarchical_topics = topic_model.hierarchical_topics(docs)
    
    # BINARY: Extract BOTH children (left and right)
    parent_child_map = {}
    
    for _, row in hierarchical_topics.iterrows():
        parent_id = int(row['Parent_ID'])
        children = []
        
        # Left child
        if pd.notna(row['Child_Left_ID']):
            children.append(int(row['Child_Left_ID']))
        
        # Right child
        if pd.notna(row['Child_Right_ID']):
            children.append(int(row['Child_Right_ID']))
        
        if children:
            parent_child_map[parent_id] = children
    
    # Rest of function...
    topics_over_time = topic_model.topics_over_time(
        docs, timestamps,
        global_tuning=global_tuning,
        evolution_tuning=evolution_tuning,
        nr_bins=len(set(timestamps))
    )
    
    parent_topics = list(parent_child_map.keys())
    parent_topics_over_time = topics_over_time[
        topics_over_time['Topic'].isin(parent_topics)
    ].copy() if len(parent_topics) > 0 else None
    
    return {
        'all_topics_over_time': topics_over_time,
        'parent_topics_over_time': parent_topics_over_time,
        'parent_child_mapping': parent_child_map,
        'hierarchical_structure': hierarchical_topics,
        'parent_topic_ids': parent_topics
    }



def visualize_parent_topics_over_time(
    topic_model,
    topics_over_time_dict,
    top_n_parents=10,
    use_openai_labels=True,
    use_all_topics=False  # ← NEW
):
    import plotly.graph_objects as go
    
    # ← CHOOSE DATASET
    if use_all_topics or len(topics_over_time_dict['parent_topics_over_time']) == 0:
        topics_df = topics_over_time_dict['all_topics_over_time']
    else:
        topics_df = topics_over_time_dict['parent_topics_over_time']
    
    if topics_df is None or len(topics_df) == 0:
        return None  # ← Prevents AttributeError
    
    # Rest of function...
    top_topics = (
        topics_df
        .groupby('Topic')['Frequency']
        .sum()
        .nlargest(top_n_parents)
        .index.tolist()
    )
    
    fig = go.Figure()
    for topic_id in sorted(top_topics):
        topic_data = topics_df[topics_df['Topic'] == topic_id]
        label = get_topic_label(topic_model, topic_id,use_custom_labels = True)
        
        fig.add_trace(go.Scatter(
            x=topic_data['Timestamp'],
            y=topic_data['Frequency'],
            mode='lines+markers',
            name=f"Topic {topic_id}: {label}",
            line=dict(width=2.5),
            marker=dict(size=8)
        ))
    
    fig.update_layout(
        title="<b>Topics Evolution Over Time</b>",
        xaxis_title="Year",
        yaxis_title="Frequency",
        template="plotly_white",
        height=600,
        width=1200
    )
    
    return fig


 

In [73]:
topic_info = topic_model.get_topic_info()
openai_labels = dict(zip(topic_info['Topic'], topic_info['OpenAI']))
topic_model.set_topic_labels(openai_labels)
 # 5. Apply topics_over_time to hierarchical
print("Applying topics_over_time to hierarchical structure...")
temporal_hierarchical = apply_topics_over_time_to_hierarchical(topic_model, docs, timestamps, hierarchical_topics)


Applying topics_over_time to hierarchical structure...


In [87]:

# 6. Visualizations
print("Creating visualizations...")
fig_parent_timeline = visualize_parent_topics_over_time(
    topic_model, temporal_hierarchical, top_n_parents=10, use_all_topics  = True
)
#fig_parent_timeline.write_html("parent_topics_timeline.html")
fig_parent_timeline

Creating visualizations...


In [66]:
fig_parent_timeline

In [58]:
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

In [7]:
tree = topic_model.get_topic_tree(hierarchical_topics)
print(tree)

.
├─lalalala_laï_lala_lalala_la
│    ├─■──gunz_yaw_big_pop_fumer ── Topic: 99
│    └─■──lalalala_laï_lala_lalala_la ── Topic: 98
└─la_de_est_les_le
     ├─elle_veut_son_qu_lui
     │    ├─■──elle_veut_son_la_aime ── Topic: 92
     │    └─elle_qu_lui_veut_son
     │         ├─elle_qu_lui_son_veut
     │         │    ├─■──elle_lui_son_je_qu ── Topic: 30
     │         │    └─■──elle_veut_qu_dit_son ── Topic: 24
     │         └─■──elle_la_suis_dans_est ── Topic: 54
     └─la_de_les_le_est
          ├─de_je_la_est_que
          │    ├─tu_je_moi_toi_pas
          │    │    ├─■──où_non_pourquoi_bla_allô ── Topic: 96
          │    │    └─tu_je_moi_toi_pas
          │    │         ├─tu_je_moi_toi_pas
          │    │         │    ├─■──bella_elle_oh_chica_ma ── Topic: 90
          │    │         │    └─tu_je_toi_moi_que
          │    │         │         ├─tu_je_toi_moi_que
          │    │         │         │    ├─tu_pas_est_qu_je
          │    │         │         │    │    ├─tu_pas_elle_mo

In [ ]:
new_topics = topic_model.reduce_outliers(
    documents=docs,
    topics=topics,
    strategy="embeddings",      # uses your mpnet embeddings
    embeddings=embeddings,
    threshold=0.1               # 0.0 => force every outlier into the closest topic
)

topic_model.update_topics(docs, topics=new_topics)

topic_info = topic_model.get_topic_info()



In [10]:
topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,0,77124,0_la_pas_les_le,"[la, pas, les, le, tu, est, dans, de, on, ai]","[Hum, eh\nEh, yah, eh, eh, eh, yah\nEh, eh, eh..."
1,1,376,1_école_étais_ai_cours,"[école, étais, ai, cours, des, prof, les, teen...","[Quand j'étais jeune, j'étais pas bête mais j'..."
2,2,413,2_han_chinois_ekip_comme,"[han, chinois, ekip, comme, bloquer, asiatique...","[Vous voulez faire le million ? Les gars, fait..."
3,3,345,3_aime_scalap_quand_shit,"[aime, scalap, quand, shit, fumer, les, adore,...",[J'aime les meufs qui aiment les meufs\nJ'fais...
4,4,328,4_histoire_il_elle_une,"[histoire, il, elle, une, était, un, son, se, ...",[C'est l'histoire d'un gars du Sept-Cinq qui t...
...,...,...,...,...,...
248,248,85,248_pyramide_salope_horizontale_aller,"[pyramide, salope, horizontale, aller, ira, mm...","[On ira où tu voudras, quand tu voudras (pour ..."
249,249,32,249_temps_time_avons_parler,"[temps, time, avons, parler, non, trop, anomal...","[Nous n'avons pas le temps\nPas le temps, non\..."
250,250,40,250_bleu_yeux_ciel_maman,"[bleu, yeux, ciel, maman, eux, rêve, barrer, b...",[J'veux qu'elle puisse regarder l'ciel et y vo...
251,251,176,251_bum_rihanna_yeah_samantha,"[bum, rihanna, yeah, samantha, sianna, elle, e...",[Réponds-moi juste une fois (juste une fois)\n...


In [11]:
topic_model.visualize_topics()

In [13]:
topic_distr, _ = topic_model.approximate_distribution(docs)

In [14]:
topic_model.visualize_distribution(topic_distr[1])

In [ ]:
topic_distr, topic_token_distr = topic_model.approximate_distribution(docs, calculate_tokens=True)
test = topic_model.visualize_approximate_distribution(docs[1], topic_token_distr[1])
test

In [11]:
import spacy
from typing import List

class ImprovedTokenizer:
    def __init__(self):
        self.nlp = spacy.load("fr_core_news_sm", disable=["parser", "ner"])
        self.MEANINGFUL_BIGRAMS = {
          ("ADJ", "NOUN"),
          ("NOUN", "NOUN"),
          ("VERB", "NOUN"),
          ("PRON","NOUN")
          }
    # Keep only the most meaningful syntactic bigram patterns
    def __call__(self, text: str) -> List[str]:
        doc = self.nlp(text[:1000000]) # truncate long docs for speed
        tokens = [(t.text, t.lemma_.lower(), t.pos_) for t in doc if t.is_alpha]
        bigrams = []
        for i in range(len(tokens) - 1):
            word1, lemma1, pos1 = tokens[i]
            word2, lemma2, pos2 = tokens[i + 1]
            if (pos1, pos2) in self.MEANINGFUL_BIGRAMS:
            # Optionally lowercase both words to normalize
                bigrams.append(f"{lemma1} {lemma2}")
        return bigrams

In [13]:
vectorizer_model= CountVectorizer(
            tokenizer= #ImprovedTokenizer(),
            stop_words=list(SPACY_STOPWORDS),
            min_df=5,  # Ignore terms that appear in < 5 docs
            max_df=0.9  # Ignore terms that appear in > 95% docs
        )

topic_model.update_topics(docs, vectorizer_model= vectorizer_model)
topic_model.get_topic_info()

SyntaxError: invalid syntax (1552404961.py, line 3)

In [ ]:
from bertopic_french_rap import BERTopicFrenchRap

model = BERTopicFrenchRap(
        embedding_model='solon',
        preprocessing='light',
        min_cluster_size=15
    )

model.load_data().preprocess().fit()

#hierarchical = model.get_hierarchical_topics()

# note pour plus tard, remove more than 95% of chare word document
#restart with small model and test each step.

In [6]:
model.print_topics(top_n=15)


TOPICS (Top 15)

Topic -1 (Outliers): 64 documents

Topic 0: 0_faire_vie_ouais_jamais
  Documents: 36863
  Top terms: faire (0.030), vie (0.027), ouais (0.026), jamais (0.019), rap (0.016), gros (0.016), veut (0.016), cœur (0.014), fois (0.013), voir (0.013)

Topic 1: 1_truc_fout_vite_style
  Documents: 63
  Top terms: truc (0.222), fout (0.058), vite (0.039), style (0.036), têtes (0.032), trucs (0.028), tape (0.025), mec (0.025), musique (0.024), danse (0.022)

Topic 2: 2_baise_nique_pute_bah
  Documents: 62
  Top terms: baise (0.066), nique (0.047), pute (0.044), bah (0.031), jamais (0.029), faire (0.028), ouais (0.026), femmes (0.024), rap (0.022), eau (0.020)

Topic 3: 3_gueule_gros_zéro_veux
  Documents: 51
  Top terms: gueule (0.115), gros (0.077), zéro (0.047), veux (0.040), soir (0.031), rêve (0.031), ouais (0.030), midi (0.028), maman (0.026), compte (0.020)

Topic 4: 4_oh oh_oh__
  Documents: 32
  Top terms: oh oh (1.313), oh (1.198),  (0.000),  (0.000),  (0.000),  (0.000), 

In [ ]:
fig = model.visualize_hierarchy()
fig.show()

model.print_topics(top_n=15)

In [5]:
def minimal_cleaning(text):
    """
    Minimal cleaning for French rap lyrics.
    Keep verlan, slang, borrowings - they're semantically important!
    Only remove technical artifacts.
    
    Based on research: rap lyrics contain 93% standard French,
    and NSL elements (verlan, borrowings) are identity markers.
    """
    if pd.isna(text) or text == '':
        return ''
    
    # Remove common lyrical artifacts that add noise
    text = re.sub(r'\[.*?\]', '', text)  # Remove [Intro], [Verse 1], etc.
    text = re.sub(r'\(.*?\)', '', text)  # Remove (x2), (Refrain), etc.
    
    # Remove excess whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Keep everything else: verlan, slang, borrowings, vulgarities
    # These are semantically meaningful in rap context
    
    return text

def split_into_verses(lyrics, min_length=50):
    """
    Split lyrics into verses (couplets) based on line breaks.
    Rap typically has clear verse structure.
    
    Args:
        lyrics: Full song lyrics
        min_length: Minimum characters for a valid verse (filter out short intros)
    
    Returns:
        List of verse strings
    """
    if pd.isna(lyrics) or lyrics == '':
        return []
    
    # Split on double line breaks (typical verse separator)
    verses = re.split(r'\n\s*\n', lyrics)
    
    # Filter out very short segments (likely artifacts)
    verses = [v.strip() for v in verses if len(v.strip()) >= min_length]
    
    return verses

print("\nCleaning lyrics...")
df['lyrics_clean'] = df['lyrics'].apply(minimal_cleaning)

# Remove empty lyrics
df = df[df['lyrics_clean'].str.len() > 0].copy()
print(f"After cleaning: {len(df)} songs remaining")


Cleaning lyrics...
After cleaning: 37298 songs remaining


In [15]:
# Comprehensive French stopwords
french_stopwords = [
    'le', 'la', 'les', 'l', 'un', 'une', 'des', 'du', 'de',
    'je', 'tu', 'il', 'elle', 'on', 'nous', 'vous', 'ils', 'elles',
    'me', 'te', 'se', 'lui', 'leur', 'moi', 'toi',
    'ce', 'cet', 'cette', 'ces', 'ça', 'c',
    'mon', 'ton', 'son', 'ma', 'ta', 'sa', 'mes', 'tes', 'ses',
    'à', 'au', 'aux', 'en', 'dans', 'sur', 'avec', 'sans', 'pour', 'par',
    'et', 'ou', 'mais', 'donc', 'car', 'ni', 'que', 'qu',
    'qui', 'quoi', 'dont', 'où',
    'être', 'suis', 'es', 'est', 'sommes', 'êtes', 'sont',
    'avoir', 'ai', 'as', 'a', 'avons', 'avez', 'ont',
    'ne', 'n', 'pas', 'plus', 'jamais', 'rien',
    'très', 'bien', 'mal', 'aussi', 'trop', 'peu', 'tout',
    'oui', 'non', 'si', 'voilà',
    'yeah', 'yo', 'oh', 'ah', 'eh', 'ok', 'hey', 'ouais', 'ben'
]

vectorizer_model = CountVectorizer(
    stop_words=french_stopwords,
    min_df=3,  # Word must appear in at least 3 documents
    max_df=0.85,  # Ignore words in >85% of documents
    ngram_range=(1, 2),  # Unigrams and bigrams
    lowercase=True
)

In [23]:
from umap import UMAP
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer


from nltk.corpus import stopwords


# Step 1 - Extract embeddings
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Step 2 - Reduce dimensionality
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine')

# Step 3 - Cluster reduced embeddings
hdbscan_model = HDBSCAN(min_cluster_size=15, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

# Step 4 - Tokenize topics
vectorizer_model = CountVectorizer(stop_words=stopwords.words("french"))

# Step 5 - Create topic representation
ctfidf_model = ClassTfidfTransformer()

# Step 6 - (Optional) Fine-tune topic representations with
# a `bertopic.representation` model
representation_model = KeyBERTInspired()

# All steps together
topic_model = BERTopic(
  embedding_model=embedding_model,          # Step 1 - Extract embeddings
  umap_model=umap_model,                    # Step 2 - Reduce dimensionality
  hdbscan_model=hdbscan_model,              # Step 3 - Cluster reduced embeddings
  vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
  ctfidf_model=ctfidf_model,                # Step 5 - Extract topic words
  representation_model=representation_model # Step 6 - (Optional) Fine-tune topic representations
)

In [ ]:
docs_fullsong = df['lyrics_clean'].tolist()
metadata_fullsong = df[['artist', 'title', 'year']].to_dict('records')



In [25]:
topic_model.visualize_topics()

In [6]:
docs_fullsong = df['lyrics_clean'].tolist()
metadata_fullsong = df[['artist', 'title', 'year']].to_dict('records')

sample_size = 1000
sample_indices = np.random.choice(len(docs_fullsong), size=sample_size, replace=False)
docs = [docs_fullsong[i] for i in sample_indices]
print(len(docs))

1000


In [16]:
def create_bertopic_hf_camembert(vectorizer_model):
    """
    ⚠️  EXPERIMENTAL - May have GPU issues
    Uses HuggingFace pipeline for CamemBERT
    """
    from transformers.pipelines import pipeline
    from bertopic import BERTopic

    # Try CamemBERT via transformers pipeline
    embedding_model = pipeline(
        "feature-extraction",
        model="camembert-base",
        device=0  # GPU
    )

    topic_model = BERTopic(
        embedding_model=embedding_model,
        vectorizer_model = vectorizer_model,
        language='french',
        verbose=True
    )

    return topic_model

tm  = create_bertopic_hf_camembert(vectorizer_model)
topics, probs = tm.fit_transform(docs)

Device set to use cuda:0
2025-11-19 16:17:30,991 - BERTopic - Embedding - Transforming documents to embeddings.
100%|██████████| 1000/1000 [00:40<00:00, 24.63it/s]
2025-11-19 16:18:11,609 - BERTopic - Embedding - Completed ✓
2025-11-19 16:18:11,609 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-11-19 16:18:12,868 - BERTopic - Dimensionality - Completed ✓
2025-11-19 16:18:12,869 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-11-19 16:18:12,886 - BERTopic - Cluster - Completed ✓
2025-11-19 16:18:12,888 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-11-19 16:18:13,269 - BERTopic - Representation - Completed ✓


In [17]:
tm.visualize_topics()

In [18]:
hierarchical_topics = tm.hierarchical_topics(docs)
tm.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

100%|██████████| 8/8 [00:00<00:00, 550.34it/s]


In [5]:
docs_fullsong = df['lyrics_clean'].tolist()
metadata_fullsong = df[['artist', 'title', 'year']].to_dict('records')

In [7]:
topic_model = BERTopic(verbose=True,language ="french")
topics, probs = topic_model.fit_transform(docs_fullsong)

2025-11-19 15:48:32,422 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 1166/1166 [00:58<00:00, 19.94it/s]
2025-11-19 15:49:38,118 - BERTopic - Embedding - Completed ✓
2025-11-19 15:49:38,119 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-11-19 15:49:58,717 - BERTopic - Dimensionality - Completed ✓
2025-11-19 15:49:58,719 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-11-19 15:50:01,197 - BERTopic - Cluster - Completed ✓
2025-11-19 15:50:01,202 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-11-19 15:50:08,255 - BERTopic - Representation - Completed ✓


In [8]:
hierarchical_topics = topic_model.hierarchical_topics(docs_fullsong)

100%|██████████| 11/11 [00:00<00:00, 144.45it/s]


In [9]:
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

In [6]:
print("\nLoading CamemBERT embedding model...")
embedding_model = SentenceTransformer("Lajavaness/sentence-camembert-large",  device='cuda:0')


Loading CamemBERT embedding model...


In [8]:
vectorizer_model = CountVectorizer(
    #stop_words=french_stopwords,
    min_df=3,  # Word must appear in at least 3 documents
    max_df=0.85,  # Ignore words in >85% of documents
    ngram_range=(1, 2),  # Unigrams and bigrams
    lowercase=True
)

In [9]:
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)

In [10]:
hdbscan_model_fullsong = HDBSCAN(
    min_cluster_size=15,  # Full songs, can have smaller clusters
    min_samples=5,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

In [11]:
print("\n" + "="*70)
print("APPROACH A: Hierarchical Topic Modeling on FULL SONGS")
print("="*70)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model_fullsong,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    #representation_model=representation_model,
    language='french',
    calculate_probabilities=True,
    low_memory=True,
    verbose=True
)


APPROACH A: Hierarchical Topic Modeling on FULL SONGS


In [12]:
sample_size = 1000
sample_indices = np.random.choice(len(docs_fullsong), size=sample_size, replace=False)
sample_docs = [docs_fullsong[i] for i in sample_indices]



In [13]:
topics_fullsong, probs_fullsong = topic_model_fullsong.fit_transform(sample_docs)

2025-11-19 15:22:47,193 - BERTopic - Embedding - Transforming documents to embeddings.
Batches:   0%|          | 0/32 [00:00<?, ?it/s]/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [512,0,0], thread: [224,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [512,0,0], thread: [225,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [512,0,0], thread: [226,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernelUtils.cu:16: vectorized_gather_kernel: block: [512,0,0], thread: [227,0,0] Assertion `ind >=0 && ind < ind_dim_size && "vectorized gather kernel index out of

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
